# SBOM-to-Audit — Stage 5 Colab Checkpoint

This notebook independently verifies Stage 5 Rapid Pivot from a clean GitHub checkout and an isolated Python environment.

It preserves the established checkpoint workflow and packages the exact commit, release report, deterministic outputs, pilot paper assets, and environment metadata into a downloadable evidence bundle.


In [ ]:
REPO_URL = "https://github.com/richietrap/sbom_to_audit.git"
REF = "main"  # Replace with a Stage 5 branch or tag when appropriate.
print("Repository:", REPO_URL)
print("Reference:", REF)


In [ ]:
import csv
import json
import os
import platform
import shutil
import subprocess
import sys
from pathlib import Path

WORKDIR = Path("/content/sbom_to_audit")
VENV_DIR = Path("/content/sbom_to_audit_stage5_venv")
for path in (WORKDIR, VENV_DIR):
    if path.exists():
        shutil.rmtree(path)
subprocess.run(
    ["git", "clone", "--depth", "1", "--branch", REF, REPO_URL, str(WORKDIR)],
    check=True,
)
os.chdir(WORKDIR)
COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("Commit:", COMMIT)
print("Kernel Python:", sys.version)
print("Platform:", platform.platform())


In [ ]:
try:
    subprocess.run([sys.executable, "-m", "venv", str(VENV_DIR)], check=True)
except subprocess.CalledProcessError:
    subprocess.run([sys.executable, "-m", "pip", "install", "virtualenv"], check=True)
    subprocess.run([sys.executable, "-m", "virtualenv", str(VENV_DIR)], check=True)
PYTHON = VENV_DIR / "bin" / "python"
subprocess.run(
    [str(PYTHON), "-m", "pip", "install", "--upgrade", "pip", "setuptools", "wheel"],
    check=True,
)
subprocess.run(
    [str(PYTHON), "-m", "pip", "install", "--no-cache-dir", "-e", ".[dev]"],
    check=True,
)
subprocess.run([str(PYTHON), "-m", "pip", "check"], check=True)
print("PASS: isolated dependency integrity")


In [ ]:
RELEASE_REPORT = Path("/content/stage5_colab_release_validation.json")
subprocess.run(
    [str(PYTHON), "scripts/release_check.py", "--report", str(RELEASE_REPORT)],
    check=True,
)
release = json.loads(RELEASE_REPORT.read_text(encoding="utf-8"))
assert release["status"] == "PASS", release["errors"]
assert len(release["deterministic_hashes"]) == 42
print("PASS: canonical release gate")
print("PASS: 42 deterministic outputs across seven scenarios")


In [ ]:
OUTPUT_ROOT = Path("/content/stage5_checkpoint_outputs")
ASSET_ROOT = Path("/content/stage5_checkpoint_paper_assets")
for path in (OUTPUT_ROOT, ASSET_ROOT):
    if path.exists():
        shutil.rmtree(path)
for scenario in sorted(Path("data/scenarios").glob("*.yaml")):
    subprocess.run(
        [
            str(PYTHON),
            "-m",
            "sbom_to_audit.cli",
            "--scenario",
            str(scenario),
            "--output-root",
            str(OUTPUT_ROOT),
        ],
        check=True,
    )
subprocess.run(
    [
        str(PYTHON),
        "paper_assets/scripts/build_stage5_assets.py",
        "--output-root",
        str(OUTPUT_ROOT),
        "--destination",
        str(ASSET_ROOT),
    ],
    check=True,
)


In [ ]:
main_pack = json.loads(
    (OUTPUT_ROOT / "evidence_packs/rapid_pivot.json").read_text(encoding="utf-8")
)
control_pack = json.loads(
    (OUTPUT_ROOT / "evidence_packs/rapid_pivot_control.json").read_text(encoding="utf-8")
)
main_metrics = json.loads(
    (OUTPUT_ROOT / "metrics/rapid_pivot_metrics.json").read_text(encoding="utf-8")
)
control_metrics = json.loads(
    (OUTPUT_ROOT / "metrics/rapid_pivot_control_metrics.json").read_text(encoding="utf-8")
)
with (OUTPUT_ROOT / "state_logs/rapid_pivot.csv").open(newline="") as handle:
    main_states = list(csv.DictReader(handle))
with (OUTPUT_ROOT / "state_logs/rapid_pivot_control.csv").open(newline="") as handle:
    control_states = list(csv.DictReader(handle))

assert [row["observed_state"] for row in main_states] == [
    "Prepare",
    "Prepare",
    "Escalate",
    "Report-Ready",
    "Report-Ready",
    "Report-Ready",
]
assert [row["observed_state"] for row in control_states] == [
    "Prepare",
    "Report-Ready",
    "Report-Ready",
    "Report-Ready",
    "Report-Ready",
    "Report-Ready",
]
assert float(main_states[0]["U_t"]) > 0.5
assert float(main_states[3]["U_t"]) == 0.15
assert main_states[2]["clock_safeguard_triggered"] == "True"
assert control_states[2]["clock_safeguard_triggered"] == "False"
assert main_states[2]["deadline_posture"] == control_states[2]["deadline_posture"]
assert main_metrics["CA"] == 1.0
assert control_metrics["CA"] is None
assert main_metrics["supplemental"]["clock_safeguard_triggers"] == 1
assert control_metrics["supplemental"]["clock_safeguard_triggers"] == 0
assert main_pack["identity_resolution"]["matching_method"] == "exact_cpe_confirmed"
assert main_pack["identity_resolution"]["gamma_id"] == 0.7
assert main_pack["decision_state"]["authorized_state"] == "Report"
assert control_pack["decision_state"]["authorized_state"] == "Report"

import yaml
main_spec = yaml.safe_load(Path("data/scenarios/rapid_pivot.yaml").read_text())
control_spec = yaml.safe_load(Path("data/scenarios/rapid_pivot_control.yaml").read_text())
assert main_spec["source_catalog"] == control_spec["source_catalog"]
assert main_spec["deadline_profile"] == control_spec["deadline_profile"]
assert main_spec["target"] == control_spec["target"]
assert [event["timestamp"] for event in main_spec["replay_events"]] == [
    event["timestamp"] for event in control_spec["replay_events"]
]
print("PASS: unresolved uncertainty enters Prepare")
print("PASS: 18-hour internal safeguard triggers Escalate")
print("PASS: early-resolution control avoids clock escalation")
print("PASS: source bytes, target, deadline profile, and event clock are matched")


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(ASSET_ROOT / "figures/rapid_pivot_clock_comparison.svg")))


In [ ]:
import datetime as dt
import hashlib
import zipfile

environment = {
    "checked_at_utc": dt.datetime.now(dt.timezone.utc).isoformat(),
    "repository": REPO_URL,
    "reference": REF,
    "commit": COMMIT,
    "kernel_python": sys.version,
    "isolated_python": subprocess.check_output([str(PYTHON), "--version"], text=True).strip(),
    "platform": platform.platform(),
    "release_status": release["status"],
    "scenarios": [
        "ghost_logger",
        "false_comfort",
        "false_comfort_control",
        "operational_outlier",
        "operational_outlier_control",
        "rapid_pivot",
        "rapid_pivot_control",
    ],
}
ENV_PATH = Path("/content/stage5_colab_environment.json")
ENV_PATH.write_text(json.dumps(environment, indent=2) + "\n", encoding="utf-8")
BUNDLE = Path("/content/stage5_colab_checkpoint_evidence.zip")
with zipfile.ZipFile(BUNDLE, "w", zipfile.ZIP_DEFLATED) as archive:
    archive.write(RELEASE_REPORT, RELEASE_REPORT.name)
    archive.write(ENV_PATH, ENV_PATH.name)
    for root in (OUTPUT_ROOT, ASSET_ROOT):
        for path in sorted(root.rglob("*")):
            if path.is_file():
                archive.write(path, f"{root.name}/{path.relative_to(root)}")
digest = hashlib.sha256(BUNDLE.read_bytes()).hexdigest()
print("Created:", BUNDLE)
print("Tested Git commit:", COMMIT)
print("Colab evidence bundle SHA-256:", digest)
print("Download the ZIP and preserve both values for checkpoint registration.")
